# Heart Disease Prediction using Machine Learning

**Goal:** Predict whether a patient has heart disease based on clinical attributes (age, cholesterol, blood pressure, chest pain type, etc.)

**Dataset:** UCI Heart Disease Dataset (Cleveland database), 303 records, 13 features + target.

**Type:** Binary Classification

**Pipeline:**
1. Load & explore data
2. Clean & preprocess
3. Exploratory Data Analysis (EDA)
4. Train/test split & scaling
5. Train multiple models (Logistic Regression, Random Forest, KNN)
6. Evaluate & compare models
7. Conclusion

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    roc_auc_score, roc_curve
)

sns.set(style="whitegrid")
%matplotlib inline

## 2. Load the Dataset

Two options are given below — use whichever works in your environment (Colab/Jupyter with internet access). Only run **one** of the two cells.

In [ ]:
# Option A: Official UCI ML Repository package (recommended)
# !pip install ucimlrepo

from ucimlrepo import fetch_ucirepo

heart_disease = fetch_ucirepo(id=45)  # Heart Disease dataset
X = heart_disease.data.features
y = heart_disease.data.targets

df = pd.concat([X, y], axis=1)
print(heart_disease.metadata.abstract if hasattr(heart_disease.metadata, 'abstract') else '')
df.head()

In [ ]:
# Option B: Load directly from a CSV mirror (use if ucimlrepo isn't available)
# url = "https://raw.githubusercontent.com/sharmaroshan/Heart-UCI-Dataset/master/heart.csv"
# df = pd.read_csv(url)
# df.head()

## 3. Understand the Columns

| Column | Meaning |
|---|---|
| age | Age in years |
| sex | 1 = male, 0 = female |
| cp | Chest pain type (0-3) |
| trestbps | Resting blood pressure (mm Hg) |
| chol | Serum cholesterol (mg/dl) |
| fbs | Fasting blood sugar > 120 mg/dl (1 = true) |
| restecg | Resting ECG results (0-2) |
| thalach | Max heart rate achieved |
| exang | Exercise-induced angina (1 = yes) |
| oldpeak | ST depression induced by exercise |
| slope | Slope of peak exercise ST segment |
| ca | Number of major vessels colored by fluoroscopy (0-3) |
| thal | 3 = normal, 6 = fixed defect, 7 = reversible defect |
| target / num | 0 = no heart disease, >0 = heart disease present |

In [ ]:
df.info()
df.describe()

## 4. Data Cleaning

In [ ]:
# Check missing values
print(df.isnull().sum())

# The 'num' target from ucimlrepo ranges 0-4 (severity). Convert to binary: 0 = no disease, 1 = disease present
target_col = 'num' if 'num' in df.columns else 'target'
df[target_col] = df[target_col].apply(lambda x: 1 if x > 0 else 0)

# Drop rows with missing values (dataset has a handful of '?' in ca/thal columns)
df = df.dropna()

# Remove duplicate rows if any
df = df.drop_duplicates()

df[target_col].value_counts()

## 5. Exploratory Data Analysis (EDA)

In [ ]:
# Target distribution
sns.countplot(x=target_col, data=df)
plt.title('Heart Disease Distribution (0 = No, 1 = Yes)')
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Feature Correlation Heatmap')
plt.show()

In [ ]:
# Age vs heart disease
sns.histplot(data=df, x='age', hue=target_col, kde=True, multiple='stack')
plt.title('Age Distribution by Heart Disease Status')
plt.show()

## 6. Train/Test Split & Feature Scaling

In [ ]:
X = df.drop(columns=[target_col])
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Training samples: {X_train.shape[0]}, Test samples: {X_test.shape[0]}')

## 7. Model Training

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=7)
}

results = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    results[name] = acc
    print(f'--- {name} ---')
    print(f'Accuracy: {acc:.4f}')
    print(classification_report(y_test, y_pred))
    print()

## 8. Model Comparison

In [ ]:
results_df = pd.DataFrame(list(results.items()), columns=['Model', 'Accuracy']).sort_values('Accuracy', ascending=False)
print(results_df)

sns.barplot(x='Accuracy', y='Model', data=results_df)
plt.title('Model Accuracy Comparison')
plt.xlim(0, 1)
plt.show()

## 9. Confusion Matrix (Best Model)

In [ ]:
best_model_name = results_df.iloc[0]['Model']
best_model = models[best_model_name]
y_pred_best = best_model.predict(X_test_scaled)

cm = confusion_matrix(y_test, y_pred_best)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Disease', 'Disease'],
            yticklabels=['No Disease', 'Disease'])
plt.title(f'Confusion Matrix - {best_model_name}')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

## 10. Feature Importance (Random Forest)

In [ ]:
rf_model = models['Random Forest']
importances = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x=importances.values, y=importances.index)
plt.title('Feature Importance (Random Forest)')
plt.xlabel('Importance')
plt.show()

## 11. Conclusion

- Summarize which model performed best and why.
- Mention the most important features (typically `cp`, `thalach`, `oldpeak`, `ca`, `thal` are strong predictors).
- Discuss limitations (small dataset size, possible class imbalance, features that may need more medical context).
- Suggest future improvements (hyperparameter tuning with GridSearchCV, trying XGBoost/SVM, cross-validation, collecting more data).